# RAGAS Testing
The purpose of this notebook is to test a given RAG model with hard metrics (in this example, the weaviate collection "LangChain_9787ec4b92d3438a8de3ff04ead7ead6"). The user will need to run this notebook in the LSST cloud, and should have a .env file in the same directory with the variables WEAVIATE_API_KEY, OPENAI_API_KEY, HTTP_HOST, and GRPC_HOST appropriately set. The first cell creates a single question-answer RAG bot in similar to the one found in rubin_rag. The next cell runs the ragas evaluation.

In [ ]:
! pip install ragas==0.2.15

In [ ]:
import os
from dotenv import load_dotenv
import warnings
import pandas as pd

import weaviate
from weaviate.classes.init import Auth

from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.vectorstores.base import VectorStoreRetriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_weaviate.vectorstores import WeaviateVectorStore

from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    LLMContextRecall,
    ContextRelevance,
    LLMContextPrecisionWithReference,
    Faithfulness,
    FactualCorrectness,
    AnswerRelevancy,
)

In [ ]:
# Suppress all warnings
warnings.filterwarnings("ignore")

# Load environment variables
load_dotenv()

## Create RAG bot

In [ ]:
# Initialize Weaviate retriever
def configure_retriever(client) -> VectorStoreRetriever:
    """Configure the Weaviate retriever for a given collection."""

    vectorstore = WeaviateVectorStore(
        client=client,
        index_name="LangChain_9787ec4b92d3438a8de3ff04ead7ead6",
        text_key="page_content",
        embedding=OpenAIEmbeddings(),
        attributes=["source", "source_key"],
    )
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3, "return_metadata": ["score"]}
        )

# Create a question answer chain
def create_qa_chain(
    input_retriever: VectorStoreRetriever,
) -> ChatPromptTemplate:
    """Create a QA chain for the chatbot."""
    # Setup ChatOpenAI (Language Model)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)

    # Define the system message template
    system_template = """You are Rubin AI Assistant, a helpful assistant at
    Vera C Rubin Observatory.
    Do your best to answer the questions in as much detail as possible.
    Do not attempt to provide an answer if you do not know the answer.
    In your response, do not recommend reading elsewhere.
    Use the following pieces of context to answer the user's
    question at the end.
    ----------------
    {context}
    ----------------"""

    # Create a ChatPromptTemplate for the QA conversation
    qa_prompt = ChatPromptTemplate.from_messages(
        [
            SystemMessagePromptTemplate.from_template(system_template),
            MessagesPlaceholder("chat_history"),
            HumanMessagePromptTemplate.from_template("Question:```{input}```"),
        ]
    )
        # Create the QA chain
    question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
    return create_retrieval_chain(input_retriever, question_answer_chain)

## Run ragas Evaluation
The ragas evaluation creates a dataset based on questions and answers defined by the user, and then retrieves 6 metrics. These metrics are:

**Retrieval Metrics**

1. LLM Context Recall - Relevant contexts retrieved / total relevant contexts
2. Context Relevance - Is context relevant to question
3. LLM Context Precision With Reference - Are the relevant chunks ranked higher

**Generative Metrics**

5. Faithfulness - Does the LLM stick to the context
6. Factual Correctness - Is the LLM's answer factually correct
7. Answer Relevancy - Is the LLM's answer relevant to the question

The ragas library uses a secondary LLM to calculate these metrics. All metrics are scores between 0 and 1, with 1 corresponding to a perfectly performing RAG bot.

In [ ]:
try:
    # Connect to the weaviate client
    client = weaviate.connect_to_custom(
        http_host=os.getenv("HTTP_HOST"),
        http_port=8080,
        http_secure=False,
        grpc_host=os.getenv("GRPC_HOST"),
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(os.getenv("WEAVIATE_API_KEY")),
        headers={"X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")},
        skip_init_checks=True,
    )

    # Configure the chatbot
    retriever = configure_retriever(client)
    qa_chain = create_qa_chain(retriever)

    # Define sample questions and their correct answers
    sample_queries = [
        "How much of the sky, in square degrees, is the LSST system capable of imaging in a single filter in three clear nights?",
        "What is estimated to be the final processed data size over the next 10 years of LSST operations?",
        "What four facilities will the LSST data management system span?",
        "What are the 12 most important interlocking constraints on data and system properties placed by the four main science themes for Rubin Observatory?",
        "How many strongly lensed Type Ia supernovae is LSST expected to discover?",
    ]
    expected_responses = [
        "The LSST system is capable of imaging about 10,000 square degrees of sky in a single filter in three clear nights.",
        "Over the ten years of LSST operations and 11 data releases, this processing will result in a cumulative processed data size approaching 500 petabytes (PB) for imaging, and over 50 PB for the catalog databases.",
        "The LSST Data Management (DM) system will span four key facilities on three continents: the Summit Facility on Cerro Pach\u00f3n in Chile (where the initial detector cross-talk correction will be performed); the Base Facility in La Serena, Chile (which will serve as a retransmission for data uploads to North America, as well as the Data Access Center for the Chilean community); the Data Processing and Archiving Facility at the National Center for Supercomputing Applications (NCSA) in Champaign-Urbana, IL; and the Satellite Processing Facility at CC-IN2P3 in Lyon, France.",
        "Here we summarize the dozen or so most important interlocking constraints on data and system properties placed by the four main science themes: 1. The depth of a single visit to a given field; 2. Image quality; 3. Photometric accuracy; 4. Astrometric accuracy; 5. Optimal exposure time; 6. The filter complement; 7. The distribution of revisit times (i.e., the cadence of observations), including the survey lifetime; 8. The total number of visits to a given area of sky; 9. The coadded survey depth; 10. The distribution of visits on the sky, and the total sky coverage; 11. The distribution of visits per filter; and 12. Parameters characterizing data processing and data access (such as the maximum time allowed after each exposure to report transient sources, and the maximum allowed software contribution to measurement errors).",
        "LSST will discover between 500 and 1000 strongly lensed Type Ia supernovae."
    ]

    dataset = []
    condensed_dataset = []

    # Create the dataset
    for query, reference in zip(sample_queries, expected_responses):
        result = qa_chain.invoke({
            "input": query,
            "chat_history": [],
        })
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": [doc.page_content for doc in result["context"]],
                "response": result["answer"],
                "reference": reference
            }
        )
        condensed_dataset.append(
            {
                "user_input": query,
                "response": result["answer"],
                "reference": reference
            }
        )

    # Evaluate the dataset
    evaluation_dataset = EvaluationDataset.from_list(dataset)

    llm = ChatOpenAI(model="gpt-4o-mini")
    evaluator_llm = LangchainLLMWrapper(llm)

    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[
            LLMContextRecall(), # Relevant contexts retrieved/ total relevant contexts
            ContextRelevance(), # Is context relevant to question
            LLMContextPrecisionWithReference(), # Are the relevant chunks ranked higher
            Faithfulness(), # Does the LLM stick to the context
            FactualCorrectness(), # Is the LLM's answer factually correct
            AnswerRelevancy(), # Is the LLM's answer relevant to the question
        ],
        llm=evaluator_llm,
    )

    df = pd.DataFrame([
        {
            "question": item["user_input"],
            "llm_answer": item["response"],
            "context_recall": result["context_recall"][i],
            "context_relevance": result["nv_context_relevance"][i],
            "context_precision": result["llm_context_precision_with_reference"][i],
            "faithfulness": result["faithfulness"][i],
            "factual_correctness": result["factual_correctness(mode=f1)"][i],
            "answer_relevancy": result["answer_relevancy"][i],
        }
        for i, item in enumerate(condensed_dataset)
    ])

except Exception as e:
    print(f"An error occurred: {e}")
finally:
    client.close()

### Display output in pandas dataframe

In [ ]:
pd.set_option('display.max_colwidth', None)
df